# World Monitor-style public endpoint explorer

This notebook explores original provider APIs rather than World Monitor's private relay. Start with USGS, NASA EONET, and GDELT; FIRMS and OpenSky are optional.

In [ ]:
%pip -q install pandas requests

import os
from getpass import getpass

import pandas as pd
import requests

SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'project-3-colab-endpoint-explorer/0.1'})

def get_json(url, *, params=None, headers=None):
    response = SESSION.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()


## 1. USGS: recent M4.5+ earthquakes

This GeoJSON feed is updated every minute.

In [ ]:
USGS_URL = 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_day.geojson'
usgs = get_json(USGS_URL)
earthquakes = pd.json_normalize(usgs['features'])
earthquakes[['id', 'properties.mag', 'properties.place', 'properties.time', 'properties.url']].head(20)

## 2. NASA EONET: open natural events

Try adding `category=wildfires` or `category=severeStorms` to the parameters.

In [ ]:
EONET_URL = 'https://eonet.gsfc.nasa.gov/api/v3/events'
eonet = get_json(EONET_URL, params={'status': 'open', 'limit': 50})
eonet_events = pd.DataFrame(eonet['events'])
eonet_events[['id', 'title', 'categories', 'closed', 'link']].head(20)

## 3. GDELT: recent news matching a query

This is an article-search endpoint, not a verified event feed. Change `QUERY` to test topic vocabulary before putting anything into a pipeline.

In [ ]:
GDELT_URL = 'https://api.gdeltproject.org/api/v2/doc/doc'
QUERY = '(humanitarian aid OR displacement OR conflict)'
gdelt = get_json(GDELT_URL, params={'query': QUERY, 'mode': 'ArtList', 'format': 'json', 'maxrecords': 25, 'sort': 'HybridRel'})
articles = pd.DataFrame(gdelt.get('articles', []))
articles[['title', 'seendate', 'domain', 'url']].head(25) if not articles.empty else articles

## 4. NASA FIRMS: satellite fire detections (optional)

Request a free FIRMS MAP_KEY first. Keep the bounding box small: a world query can return tens of thousands of rows.

In [ ]:
NASA_FIRMS_MAP_KEY = getpass('NASA FIRMS MAP_KEY (leave blank to skip): ')
BBOX = '34,29,37,34'  # west,south,east,north; edit for an area you are studying

if NASA_FIRMS_MAP_KEY:
    firms_url = f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{NASA_FIRMS_MAP_KEY}/VIIRS_NOAA20_NRT/{BBOX}/1'
    fires = pd.read_csv(firms_url)
    display(fires.head(20))
else:
    print('Skipped. Get a free key at https://firms.modaps.eosdis.nasa.gov/api/map_key')

## 5. OpenSky: aircraft state vectors (optional)

Anonymous access may be rate-limited. A bearer token can be supplied as `OPENSKY_ACCESS_TOKEN`; do not hardcode it in a shared notebook.

In [ ]:
OPENSKY_URL = 'https://opensky-network.org/api/states/all'
MIDDLE_EAST_BBOX = {'lamin': 29, 'lomin': 34, 'lamax': 37, 'lomax': 44}
headers = {}
if os.getenv('OPENSKY_ACCESS_TOKEN'):
    headers['Authorization'] = f"Bearer {os.environ['OPENSKY_ACCESS_TOKEN']}"

try:
    opensky = get_json(OPENSKY_URL, params=MIDDLE_EAST_BBOX, headers=headers)
    columns = ['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'true_track', 'vertical_rate', 'sensors', 'geo_altitude', 'squawk', 'spi', 'position_source']
    states = pd.DataFrame(opensky.get('states') or [], columns=columns)
    display(states[['icao24', 'callsign', 'origin_country', 'longitude', 'latitude', 'baro_altitude', 'velocity']].head(50))
except requests.HTTPError as error:
    print(f'OpenSky request failed: {error}. Try again later or configure an OAuth bearer token.')

## Next experiment

For each provider, retain the raw JSON/CSV alongside a normalized envelope (`source`, `source_event_id`, `observed_at`, `fetched_at`, `payload`). That preserves provenance before filtering, deduplication, or clustering.